# 第 7 课：上下文工程与四层记忆架构（从前缀缓存到离线“做梦”整理）
### 《现代 AI 与 Agent 系统》Colab 实战实验 · 吸收 Anthropic 2026.11《The Components of Agent Memory》

> **为什么需要这节课？**
> 大模型本身**完全不记事**——它所谓的“记忆”，不过是外围程序（Harness）在每一轮请求时重新拼好发给它的那串文本。
> 这带来两个直接的工程痛点：
> 1. **单个窗口内（Within-Session）**：对话越长、塞进去的文件和工具结果越多，不仅账单成倍暴涨，如果中途改动了前面的一丁点系统提示词，还会直接击穿**前缀缓存（Prefix Cache）**，让成本贵上 5~10 倍；
> 2. **跨会话之间（Cross-Session Memory）**：Anthropic 在《The Components of Agent Memory》中指出，真正的智能体记忆不是把所有原始聊天记录塞进向量库，而是分层治理：**原始记录（Transcripts） $\rightarrow$ 关键笔记（Memory Notes） $\rightarrow$ 结构化知识页（Wikis/Representations） $\rightarrow$ 开局自动注入的薄索引（Injected Context）**，并通过后台离线**“做梦（Dreaming / Sleep-time Consolidation）”**核对原始记录、纠错并提炼规则！

---

### 本课 3 个动手实验（先写预测 $\rightarrow$ 运行实测 $\rightarrow$ 用数字说话）
1. **实验 1（前缀缓存与只追加原则）**：对比“每轮只往后追加”与“每轮在系统提示词里塞入当前时间戳/打乱工具顺序”，算清缓存命中率与真实账单差几倍；
2. **实验 2（配对安全压缩 Compaction）**：当上下文快满时，对比“粗暴按行数截断（破坏 `tool_use_id` 配对导致报错）”与“清理旧工具大输出 + 保留关键约束摘要”；
3. **实验 3（Anthropic 四层记忆与离线“做梦 Dreaming”纠错）**：模拟 Agent 在白天会话中记下了一条带有错误假设的笔记，看后台“做梦（Dreaming）”进程如何回查原始 Transcript、修正错误笔记并把高频铁律升格进 `global.md` 注入层！


In [ ]:
# [Cell 1: 环境初始化与 Token 计费 / 记忆沙箱模拟器]
import copy
import json
import os
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple

# 模拟大模型计费标准（每百万 Token 价格，单位：美元；缓存命中读取仅需 10% 成本）
PRICE_BASE_INPUT = 3.00   # 新输入 Token: $3.0 / 1M
PRICE_CACHE_HIT  = 0.30   # 命中前缀缓存 Token: $0.3 / 1M (便宜 90%!)

def measure_prefix_cache_cost(requests: List[List[str]]) -> Dict[str, Any]:
    """
    给定连续多轮发送给模型的 token 块列表，计算每轮的前缀缓存命中量与总花费。
    只要从头开始一字不差的部分，都可以复用上一轮的 KV Cache！
    """
    prev_tokens: List[str] = []
    total_cached_tokens = 0
    total_un_cached_tokens = 0

    for req in requests:
        # 计算与上一轮请求的最长公共前缀长度 (Longest Common Prefix)
        common_len = 0
        for a, b in zip(prev_tokens, req):
            if a == b:
                common_len += 1
            else:
                break
        cached = common_len * 1000       # 每个块代表 1,000 tokens
        uncached = (len(req) - common_len) * 1000
        total_cached_tokens += cached
        total_un_cached_tokens += uncached
        prev_tokens = list(req)

    cost = (total_cached_tokens / 1e6) * PRICE_CACHE_HIT + (total_un_cached_tokens / 1e6) * PRICE_BASE_INPUT
    total_all = total_cached_tokens + total_un_cached_tokens
    hit_rate = (total_cached_tokens / total_all * 100.0) if total_all else 0.0
    return {
        "cached_tokens": total_cached_tokens,
        "uncached_tokens": total_un_cached_tokens,
        "hit_rate_pct": round(hit_rate, 1),
        "cost_usd": round(cost, 4),
    }

print("[OK] Cell 1 初始化完成！前缀缓存计费器与记忆实验底座已就绪。")


---
## 实验 1：前缀缓存（Prefix Caching）——为什么 Harness 永远只往后追加？

### 1. 为什么需要它？
很多初学者写 Agent 时，喜欢在每一轮的 `System Prompt` 第一行写上 `当前时间：2026-11-08 17:30:01`，或者用 Python 的 `set` 存放工具表导致每轮工具顺序随机变化。
殊不知，大模型的 KV Cache 是**严格按从左到右的前缀匹配**的！一旦最前面的系统提示词变了 1 个字，后面哪怕有 50,000 个 Token 的历史对话一字未改，也**全部无法命中缓存**，必须全额重新计算并付 10 倍的钱！

### 2. 先写下你的预测
假设连续对话 10 轮，系统提示与工具表占 10 个块（10k tokens），每轮新增 2 个块的对话：
- **做法 A（Append-Only 只往后追加）**：缓存命中率大概是多少？
- **做法 B（每轮在 System Prompt 开头更新时间）**：缓存命中率是多少？账单会比做法 A 贵多少倍？


In [ ]:
# [Cell 2: 实验 1 — 实测 Append-Only vs 破坏前缀的缓存命中率与账单差距]

# 构造 10 轮请求
# 做法 A：固定前缀（System + Tools 固定在最前，每轮只往末尾追加新对话）
requests_append_only = []
history = [f"SYS_BLOCK_{i}" for i in range(5)] + [f"TOOL_BLOCK_{i}" for i in range(5)]
for turn in range(1, 11):
    history = history + [f"USER_TURN_{turn}", f"ASSISTANT_TURN_{turn}"]
    requests_append_only.append(list(history))

# 做法 B：每轮在最前面塞入变化的动态时间戳，导致整个前缀全部失效！
requests_mutated_prefix = []
history_b = [f"SYS_BLOCK_{i}" for i in range(5)] + [f"TOOL_BLOCK_{i}" for i in range(5)]
for turn in range(1, 11):
    history_b = history_b + [f"USER_TURN_{turn}", f"ASSISTANT_TURN_{turn}"]
    requests_mutated_prefix.append([f"DYNAMIC_TIMESTAMP_TURN_{turn}"] + list(history_b))

res_a = measure_prefix_cache_cost(requests_append_only)
res_b = measure_prefix_cache_cost(requests_mutated_prefix)

print("=== 实验 1 实测结果（连续 10 轮 Agent 对话）===")
print(f"做法 A (只往后追加 Append-Only) : 缓存命中率 = {res_a['hit_rate_pct']}% | 命中 Token = {res_a['cached_tokens']:,} | 账单 = ${res_a['cost_usd']:.4f}")
print(f"做法 B (开头插入动态时间戳)     : 缓存命中率 = {res_b['hit_rate_pct']}% | 命中 Token = {res_b['cached_tokens']:,} | 账单 = ${res_b['cost_usd']:.4f}")
print(f"💡 结论：仅仅因为在开头改了一个时间戳，账单直接贵了 {res_b['cost_usd'] / res_a['cost_usd']:.1f} 倍！")


---
## 实验 2：对话快满时的安全压缩（Compaction）——留什么、扔什么、不能拆散什么？

### 1. 为什么需要它？
当对话快把上下文窗口塞满时，我们需要压缩历史。常见的三种做法：
1. **无脑按条数截断（Naive Slice `messages[-4:]`）**：极易把上一轮的 `tool_use` 切掉、只留下孤零零的 `tool_result`（导致 API 直接报 `400 Orphaned tool_result` 错误），而且把第 1 轮用户的核心约束（例如“预算上限 2000 元、严禁修改生产库”）直接扔掉了！
2. **清理旧工具大输出 + 保留配对与核心约束摘要（Safe Compaction）**：先把早期庞大的原始文件内容替换成简短引用路径（`[Cleared 8000 chars, see path/file.py]`），并确保 `tool_use` 与 `tool_result` 永远成对保留或成对归档，同时把首轮核心约束提炼进摘要头。


In [ ]:
# [Cell 3: 实验 2 — 对比无脑截断 vs 配对安全压缩（Safe Compaction）]

RAW_CONVERSATION = [
    {"role": "user", "content": "目标：修复 config.json 端口配置。铁律约束：端口号必须在 8000~8999 之间，且不能删掉 tls 字段。"},
    {"role": "assistant", "type": "tool_use", "tool_use_id": "call_01", "name": "read_file", "content": "read logs/app.log"},
    {"role": "tool", "type": "tool_result", "tool_use_id": "call_01", "path": "logs/app.log", "content": "INFO_LOG_LINE\n" * 400},
    {"role": "assistant", "type": "tool_use", "tool_use_id": "call_02", "name": "read_file", "content": "read config.json"},
    {"role": "tool", "type": "tool_result", "tool_use_id": "call_02", "path": "config.json", "content": '{"port": 80, "tls": true}'},
    {"role": "user", "content": "现在请给出修改后的 config.json 内容。"},
]

def validate_messages_contract(messages: List[Dict[str, Any]]) -> Tuple[bool, str]:
    """模拟大模型 API 的消息合法性校验与约束保留检查。"""
    seen_tool_uses = set()
    for m in messages:
        if m.get("type") == "tool_use":
            seen_tool_uses.add(m["tool_use_id"])
        elif m.get("type") == "tool_result":
            if m["tool_use_id"] not in seen_tool_uses:
                return False, f"API Error 400: Orphaned tool_result without matching tool_use ({m['tool_use_id']})"
    full_text = " ".join(m.get("content", "") for m in messages)
    if "8000~8999" not in full_text or "tls" not in full_text:
        return False, "Constraint Lost: 丢失了首轮核心约束（端口范围 8000~8999 或 tls 约束）"
    return True, "VALID_AND_CONSTRAINTS_PRESERVED"

# 反例：无脑只取最后 4 条消息 messages[-4:]
naive_truncated = RAW_CONVERSATION[-4:]

# 🛠️ 作业任务 2：实现安全压缩函数 safe_compact_messages
def safe_compact_messages(messages: List[Dict[str, Any]], max_tool_chars: int = 120) -> List[Dict[str, Any]]:
    """
    安全压缩规则：
    1. 完整保留首条用户约束消息，以及每对 (tool_use, tool_result) 的结构；
    2. 对非最近一轮的超长 tool_result（长度 > max_tool_chars），将其 content 替换为可回查的路径占位符：
       f"[COMPACTED: original {len(content)} chars, re-read {path} if needed]"
    """
    compacted = []
    for idx, msg in enumerate(messages):
        m = copy.deepcopy(msg)
        # 如果是较早的 tool_result 且内容过长，清理大输出但保留 tool_use_id 配对和路径引用
        if m.get("type") == "tool_result" and idx < len(messages) - 2 and len(m.get("content", "")) > max_tool_chars:
            orig_len = len(m["content"])
            path = m.get("path", "unknown")
            m["content"] = f"[COMPACTED: original {orig_len} chars, re-read {path} if needed]"
        compacted.append(m)
    return compacted

compacted_msgs = safe_compact_messages(RAW_CONVERSATION)

raw_chars = sum(len(m["content"]) for m in RAW_CONVERSATION)
naive_ok, naive_reason = validate_messages_contract(naive_truncated)
safe_ok, safe_reason = validate_messages_contract(compacted_msgs)
safe_chars = sum(len(m["content"]) for m in compacted_msgs)

print("=== 实验 2 实测结果（上下文压缩对比）===")
print(f"原始消息总字符数 : {raw_chars} chars")
print(f"1. 无脑截断 messages[-4:] : 校验通过 = {naive_ok} | 原因 = {naive_reason}")
print(f"2. 安全压缩 Safe Compact  : 校验通过 = {safe_ok}  | 原因 = {safe_reason} | 压缩后字符数 = {safe_chars} chars (节省 {(1 - safe_chars/raw_chars)*100:.1f}%!)")


---
## 实验 3：Anthropic 四层记忆架构与离线“做梦（Dreaming）”整理

### 1. 为什么需要它？（吸收自 Anthropic《The Components of Agent Memory》）
当 Agent 需要跨几十个会话长期工作时，**“记下所有事情”等于“什么都没记住”**。Anthropic 将跨会话记忆拆成清晰的 **4 层结构**：
1. **第 1 层 `Transcripts`（原始记录）**：每次会话最底层的真实日志（只用于按需检索和核对事实真相，绝不整包塞进新会话）；
2. **第 2 层 `Memory Notes`（实时笔记）**：Agent 在干活时随手记下的稀疏观察（**记忆即预测**：预测未来的会话会用到这条信息）。但随手记的笔记经常会有**过时或错误的假设**；
3. **第 3 层 `Memory Representations / Wikis`（结构化知识库）**：把零散笔记整理成互相链接的专题卡片（已验证的事实、踩坑记录）；
4. **第 4 层 `Injected Context`（如 `global.md` 注入层）**：每次新会话开局**唯一自动预加载**的极薄导航索引与最高优铁律（严格限制体积）。

更重要的是 **离线“做梦（Dreaming / Sleep-time Consolidation）”机制**：
在会话结束后的空闲时间，后台启动一个整理进程，拿第 2 层的 `Memory Notes` 去和第 1 层的 `Transcripts` 原始日志对账——**删掉被证实错误的笔记、更新知识库卡片，并把多次复现的关键铁律升格写进 `global.md`**！


In [ ]:
# [Cell 4: 实验 3 — 实现四层记忆栈与离线“做梦（Dreaming）”对账巩固]

@dataclass
class FourLayerMemoryStack:
    # Layer 1: 原始会话真实记录 (Ground Truth)
    transcripts: List[Dict[str, Any]] = field(default_factory=list)
    # Layer 2: 会话中随手记下的稀疏笔记 (可能包含未经证实的错误假设)
    memory_notes: List[Dict[str, Any]] = field(default_factory=list)
    # Layer 3: 结构化专题 Wiki (按主题聚合已验证的事实与陷阱)
    wiki_pages: Dict[str, Dict[str, Any]] = field(default_factory=dict)
    # Layer 4: 每次新会话开局自动注入的薄索引与铁律 (global.md)
    injected_global_md: List[str] = field(default_factory=list)

stack = FourLayerMemoryStack(
    transcripts=[
        {"session": "s1", "event": "Ran pytest with `--workers auto`; failed with SQLite database locked error."},
        {"session": "s1", "event": "Switched to `pytest -n 0 --dist no`; all 42 SQLite tests passed."},
        {"session": "s2", "event": "Tried port 3000; blocked by local proxy. Port 8088 worked cleanly in both s1 and s2."},
    ],
    memory_notes=[
        # 注意：note_1 是 Agent 在 s1 早期随手记下的错误猜测，后来被 s1 结尾的实测推翻了！
        {"id": "note_1", "topic": "testing", "claim": "Use `--workers auto` to speed up SQLite tests.", "evidence_check": "contradicted_by_s1"},
        {"id": "note_2", "topic": "testing", "claim": "SQLite tests require serial execution (`-n 0`) to avoid lock errors.", "evidence_check": "verified_by_s1", "reused_count": 3},
        {"id": "note_3", "topic": "network", "claim": "Use port 8088 instead of 3000 for local test server.", "evidence_check": "verified_by_s2", "reused_count": 2},
    ],
)

# 🛠️ 作业任务 3：实现离线“做梦（Dreaming）”整理函数 run_memory_dreaming
def run_memory_dreaming(mem: FourLayerMemoryStack) -> Dict[str, int]:
    """
    后台离线做梦（Dreaming）三步曲：
    1. 事实核对（Audit）：对照 Transcripts 检查 memory_notes，剔除 evidence_check 为 contradicted 的错误笔记；
    2. 知识库归档（Build Wiki）：将验证通过（verified）的笔记按 topic 写入 Layer 3 (wiki_pages)；
    3. 升格注入层（Promote to Injected Context）：将复用次数 >= 2 的核心铁律提炼进 Layer 4 (injected_global_md)，保持极简！
    """
    removed_wrong_notes = 0
    kept_notes = []
    for note in mem.memory_notes:
        if str(note.get("evidence_check", "")).startswith("contradicted"):
            removed_wrong_notes += 1
            continue
        kept_notes.append(note)
        topic = note["topic"]
        if topic not in mem.wiki_pages:
            mem.wiki_pages[topic] = {"topic": topic, "verified_rules": []}
        if note["claim"] not in mem.wiki_pages[topic]["verified_rules"]:
            mem.wiki_pages[topic]["verified_rules"].append(note["claim"])
        if note.get("reused_count", 0) >= 2 and note["claim"] not in mem.injected_global_md:
            mem.injected_global_md.append(note["claim"])

    mem.memory_notes = kept_notes
    return {
        "removed_wrong_notes": removed_wrong_notes,
        "wiki_topics_count": len(mem.wiki_pages),
        "injected_rules_count": len(mem.injected_global_md),
    }

dream_stats = run_memory_dreaming(stack)
print("=== 实验 3 实测结果（Anthropic 四层记忆 + 离线 Dreaming 整理）===")
print(f"剔除的错误/过时笔记数 : {dream_stats['removed_wrong_notes']} 条 (成功拦截错误假设 `--workers auto` 污染下一次会话！)")
print(f"生成的结构化 Wiki 专题: {list(stack.wiki_pages.keys())}")
print("新会话开局自动注入的 `global.md` 铁律清单：")
for idx, rule in enumerate(stack.injected_global_md, 1):
    print(f"  [{idx}] {rule}")


In [ ]:
# [Cell 5: 自动验收与课程回执生成（Grading Receipt）]
assert res_a["hit_rate_pct"] > 75.0 and res_b["hit_rate_pct"] == 0.0, "实验 1 验收失败：前缀缓存统计异常"
assert not naive_ok and safe_ok, "实验 2 验收失败：安全压缩未能同时保住 tool_use_id 配对与首轮约束"
assert dream_stats["removed_wrong_notes"] == 1 and dream_stats["injected_rules_count"] == 2, "实验 3 验收失败：Dreaming 未能正确剔除错误笔记并升格高频规则"

receipt = {
    "lesson": "lesson-07-context-and-four-layer-memory",
    "status": "PASSED",
    "metrics": {
        "exp1_append_only_cache_hit_pct": res_a["hit_rate_pct"],
        "exp1_cost_ratio_mutated_vs_append": round(res_b["cost_usd"] / res_a["cost_usd"], 2),
        "exp2_compaction_saved_pct": round((1 - safe_chars / raw_chars) * 100, 1),
        "exp3_dreaming_contradictions_removed": dream_stats["removed_wrong_notes"],
        "exp3_injected_global_rules": dream_stats["injected_rules_count"],
    },
}
print("🎉 恭喜！第 7 课全部 3 个实验验收通过！成绩回执如下：")
print(json.dumps(receipt, indent=2, ensure_ascii=False))
